# Data Exploration: Shopping Cart Trajectories

In this notebook, we explore the shopping cart data located in `../data/raw`. The data consists of `x`, `y` coordinates, timestamps, and a quality metric `q`.

Goals:
- Explore raw data quality and drift.
- Filter out artifacts (charging stations, outside shop bounds) to assess data usability.
- Analyze high-speed anomalies (speed > 3 m/s).
- Assess positioning accuracy on stationary devices (standard deviation).

**Memory-Efficient Design**: All processing uses chunked reading (`chunksize`) and incremental
aggregation so the notebook runs on machines with 8 GB of RAM.

In [ ]:
import os
import gc
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

# ---------- constants ----------
RAW_DATA_DIR = '../data/raw'
ALL_FILES = sorted(glob.glob(os.path.join(RAW_DATA_DIR, 'node_*.csv')))
CHUNKSIZE = 100_000              # rows per chunk
SPEED_THRESHOLD = 3.0            # m/s

# Optimised dtypes – cuts memory roughly in half vs. float64/int64
DTYPE_MAP = {
    'x': 'float32',
    'y': 'float32',
    'q': 'float32',
    'node_id': 'int32',
}

print(f"Found {len(ALL_FILES)} CSV files in {RAW_DATA_DIR}")

## Helper: iterate over *all* files in chunks

The generator below yields one chunk at a time across every CSV file.
It also computes the per-node time-diff / distance / speed columns on the
fly so we never need a second pass.

In [ ]:
def iter_chunks(files=None, add_speed_cols=False):
    """
    Yield DataFrames of size <= CHUNKSIZE across all files.

    Parameters
    ----------
    files : list[str] | None
        Files to iterate; defaults to ALL_FILES.
    add_speed_cols : bool
        If True, compute time_diff / distance / speed within each chunk.
        Because diff() needs consecutive rows of the same node_id, we keep
        track of the *last row per node_id* from the previous chunk and
        **within each file** (we reset between files since node_ids don't
        span files).
    """
    if files is None:
        files = ALL_FILES

    for fpath in files:
        # Tail of previous chunk keyed by node_id (for diff continuity)
        prev_tails = {}  # node_id → Series{x, y, timestamp}

        reader = pd.read_csv(
            fpath,
            chunksize=CHUNKSIZE,
            dtype=DTYPE_MAP,
            parse_dates=['timestamp'],
        )
        for chunk in reader:
            chunk.sort_values(by=['node_id', 'timestamp'], inplace=True)

            if add_speed_cols:
                # Prepend previous tails so diff() works across chunk boundaries
                tail_rows = []
                for nid in chunk['node_id'].unique():
                    if nid in prev_tails:
                        tail_rows.append(prev_tails[nid])
                if tail_rows:
                    tail_df = pd.DataFrame(tail_rows)
                    combined = pd.concat([tail_df, chunk], ignore_index=True)
                else:
                    combined = chunk

                combined.sort_values(by=['node_id', 'timestamp'], inplace=True)
                combined['time_diff'] = combined.groupby('node_id')['timestamp'].diff().dt.total_seconds().astype('float32')
                combined['x_diff'] = combined.groupby('node_id')['x'].diff().astype('float32')
                combined['y_diff'] = combined.groupby('node_id')['y'].diff().astype('float32')
                combined['distance'] = np.sqrt(combined['x_diff']**2 + combined['y_diff']**2).astype('float32')
                combined['speed'] = (combined['distance'] / combined['time_diff']).astype('float32')

                # Drop the prepended tail rows (they were only there for diff)
                if tail_rows:
                    chunk = combined.iloc[len(tail_rows):].copy()
                else:
                    chunk = combined

                # Save last row per node_id for next chunk
                for nid, grp in chunk.groupby('node_id'):
                    prev_tails[nid] = grp.iloc[-1]

            yield chunk

## 1. Raw Data Exploration (Before Filtering)

### 1a. Missing values & descriptive statistics
We accumulate counts/sums/min/max over chunks and compute summary stats.

In [ ]:
total_rows = 0
null_counts = None
numeric_cols = ['x', 'y', 'q']
running_sum  = np.zeros(len(numeric_cols), dtype='float64')
running_sum2 = np.zeros(len(numeric_cols), dtype='float64')  # sum of squares
running_min  = np.full(len(numeric_cols), np.inf)
running_max  = np.full(len(numeric_cols), -np.inf)
running_count = np.zeros(len(numeric_cols), dtype='int64')    # non-null count

for chunk in iter_chunks():
    total_rows += len(chunk)
    nc = chunk.isnull().sum()
    if null_counts is None:
        null_counts = nc
    else:
        null_counts += nc

    for i, col in enumerate(numeric_cols):
        vals = chunk[col].dropna().values.astype('float64')
        running_count[i] += len(vals)
        running_sum[i]  += vals.sum()
        running_sum2[i] += (vals ** 2).sum()
        if len(vals):
            running_min[i] = min(running_min[i], vals.min())
            running_max[i] = max(running_max[i], vals.max())

# Build summary table
means = running_sum / running_count
stds  = np.sqrt(running_sum2 / running_count - means**2)

summary = pd.DataFrame({
    'count': running_count,
    'mean':  means,
    'std':   stds,
    'min':   running_min,
    'max':   running_max,
}, index=numeric_cols)

print(f"Raw Data Shape: ({total_rows}, columns from CSV)")
print(f"\nMissing values:\n{null_counts}")
print(f"\nData Describe:\n{summary}")

### 1b. Distribution of Quality (q) – raw data

In [ ]:
# Build a histogram of q values incrementally
q_min, q_max = running_min[numeric_cols.index('q')], running_max[numeric_cols.index('q')]
n_bins = 50
bin_edges = np.linspace(q_min, q_max, n_bins + 1)
q_hist = np.zeros(n_bins, dtype='int64')

for chunk in iter_chunks():
    vals = chunk['q'].dropna().values
    counts, _ = np.histogram(vals, bins=bin_edges)
    q_hist += counts

plt.figure(figsize=(10, 5))
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
plt.bar(bin_centers, q_hist, width=bin_centers[1] - bin_centers[0], edgecolor='black', alpha=0.7)
plt.title('Distribution of Quality (q) - Raw Data')
plt.xlabel('q')
plt.ylabel('Count')
plt.show()

### 1c. Distance between consecutive points (raw) & Drift vs Q

In [ ]:
# ---- Pass 1: find 95th percentile of distance ----
# We use a reservoir/approximate approach:
# sample ~500k distance values and take the 95th percentile.

rng = np.random.default_rng(42)
distance_sample = []
MAX_SAMPLE = 500_000

for chunk in iter_chunks(add_speed_cols=True):
    d = chunk['distance'].dropna().values
    if len(distance_sample) < MAX_SAMPLE:
        # subsample this chunk
        take = min(len(d), MAX_SAMPLE - len(distance_sample))
        idx = rng.choice(len(d), size=take, replace=False) if take < len(d) else np.arange(len(d))
        distance_sample.extend(d[idx].tolist())

distance_sample = np.array(distance_sample, dtype='float32')
p95 = np.percentile(distance_sample, 95)
print(f"95th percentile of distance: {p95:.2f}")

# ---- Build histogram of distance (zoomed to 95th percentile) ----
dist_bins = np.linspace(0, p95, 51)
dist_hist = np.zeros(50, dtype='int64')

for chunk in iter_chunks(add_speed_cols=True):
    d = chunk['distance'].dropna().values
    d = d[d < p95]
    counts, _ = np.histogram(d, bins=dist_bins)
    dist_hist += counts

plt.figure(figsize=(10, 5))
centers = 0.5 * (dist_bins[:-1] + dist_bins[1:])
plt.bar(centers, dist_hist, width=centers[1] - centers[0], edgecolor='black', alpha=0.7)
plt.title('Distribution of Distance between Consecutive Points (Raw Data, Zoomed to 95th Percentile)')
plt.xlabel('Distance')
plt.ylabel('Count')
plt.show()

In [ ]:
# ---- Scatter plot of q vs distance (sampled) ----
# Collecting ALL points would be huge; instead we sample ~200k points.
rng = np.random.default_rng(42)
scatter_q = []
scatter_d = []
MAX_SCATTER = 200_000

for chunk in iter_chunks(add_speed_cols=True):
    if len(scatter_q) >= MAX_SCATTER:
        break
    mask = chunk['distance'].notna() & chunk['q'].notna()
    sub = chunk.loc[mask, ['q', 'distance']]
    take = min(len(sub), MAX_SCATTER - len(scatter_q))
    if take < len(sub):
        sub = sub.sample(n=take, random_state=42)
    scatter_q.extend(sub['q'].values.tolist())
    scatter_d.extend(sub['distance'].values.tolist())

plt.figure(figsize=(10, 6))
plt.scatter(scatter_q, scatter_d, alpha=0.15, s=1)
plt.title('Raw Data: Scatter Plot of Quality (q) vs Distance (Drift)')
plt.xlabel('Quality (q)')
plt.ylabel('Distance')
plt.yscale('log')
plt.show()

del scatter_q, scatter_d
gc.collect();

## 2. Positioning Usability (Hyvyys/Käytettävyys) - Applying Filters
We count how many points are actual shop visits versus artifacts (out of bounds or charging stations without signal at 0,0).

In [ ]:
def classify_usability(chunk_with_speed):
    """
    Add classification columns to a chunk (in-place) and return counts dict.
    Expects 'speed' column to already exist.
    """
    c = chunk_with_speed
    c['is_charging_station'] = (c['x'] == 0) & (c['y'] == 0)

    shop_bounds = (
        ((c['x'] > 350) & (c['y'] < 3000) & (c['x'] < 1500)) |
        ((c['x'] > 1500) & (c['x'] < 8200)) |
        ((c['x'] > 8200) & (c['y'] > 450) & (c['x'] < 9650)) |
        ((c['x'] > 9650) & (c['y'] > 450) & (c['y'] < 4700) & (c['x'] < 10190))
    )
    c['in_shop'] = shop_bounds
    c['is_out_of_bounds'] = ~c['in_shop'] & ~c['is_charging_station']
    c['is_high_speed'] = c['speed'] > SPEED_THRESHOLD

    # Usability class (vectorised – avoids slow row-wise apply)
    conditions = [
        c['is_charging_station'],
        c['is_out_of_bounds'] & c['is_high_speed'],
        c['is_out_of_bounds'],
        c['is_high_speed'],
    ]
    choices = [
        'Charging Station (0,0)',
        'Out of Bounds AND >3 m/s (Extreme Noise)',
        'Out of Bounds (Noise)',
        'Inside Shop BUT >3 m/s (Speed Anomaly)',
    ]
    c['usability_class'] = np.select(conditions, choices, default='Inside Shop, Valid Speed (Usable Data)')

    return c

In [ ]:
# ---- Accumulate usability counts across all chunks ----
from collections import Counter

usability_counter = Counter()
total_pts = 0
in_shop_pts = 0
charging_pts = 0
out_pts = 0

for chunk in iter_chunks(add_speed_cols=True):
    chunk = classify_usability(chunk)
    total_pts += len(chunk)
    in_shop_pts += chunk['in_shop'].sum()
    charging_pts += chunk['is_charging_station'].sum()
    out_pts += chunk['is_out_of_bounds'].sum()

    for cls, cnt in chunk['usability_class'].value_counts().items():
        usability_counter[cls] += cnt

print(f"Total Points: {total_pts}")
print(f"- Inside Shop (Usable): {in_shop_pts}")
print(f"- Charging Station (0,0): {charging_pts}")
print(f"- Outside Shop (Noise): {out_pts}")

# ---- Pie chart: basic usability ----
labels = ['Inside Shop', 'Charging Station (0,0)', 'Outside Bounds (Noise)']
sizes  = [int(in_shop_pts), int(charging_pts), int(out_pts)]
colors = ['#4CAF50', '#FFC107', '#F44336']

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Positioning Usability')
plt.show()

## 3. High-Speed Anomalies (Speed > 3 m/s) & Usability Breakdown

In [ ]:
high_speed_pts = usability_counter.get('Inside Shop BUT >3 m/s (Speed Anomaly)', 0) + \
                 usability_counter.get('Out of Bounds AND >3 m/s (Extreme Noise)', 0)
print(f"Data points moving faster than 3 m/s: {high_speed_pts} ({high_speed_pts/total_pts*100:.2f}%)")

usability_counts = pd.Series(usability_counter).sort_values(ascending=True)
print("\nUsability Breakdown:\n", usability_counts)

plt.figure(figsize=(10, 6))
usability_counts.plot(kind='barh', color=['#4CAF50', '#F44336', '#FF9800', '#9C27B0', '#FFC107'])
plt.title('Absolute Volume of Usable vs. Anomalous Data Points')
plt.xlabel('Number of Points')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.pie(usability_counts, labels=usability_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Combined Usability, Bounds, & Speed Analysis')
plt.show()

## 4. Drift Analysis on Cleaned Data
Let's check the drift vs Q on ONLY the cleaned in-shop, valid-speed data, removing the massive leaps.

In [ ]:
# ---- Step 1: collect all q values from clean data to compute quantile bins ----
# (We only store q values, a small footprint ~4 bytes * N_clean)
# To be safe, we sample up to 2M q values to derive bin edges.

rng = np.random.default_rng(42)
clean_q_sample = []
MAX_Q_SAMPLE = 2_000_000

for chunk in iter_chunks(add_speed_cols=True):
    chunk = classify_usability(chunk)
    clean = chunk[chunk['usability_class'] == 'Inside Shop, Valid Speed (Usable Data)']
    q_vals = clean['q'].dropna().values
    if len(clean_q_sample) < MAX_Q_SAMPLE and len(q_vals) > 0:
        take = min(len(q_vals), MAX_Q_SAMPLE - len(clean_q_sample))
        if take < len(q_vals):
            idx = rng.choice(len(q_vals), size=take, replace=False)
            q_vals = q_vals[idx]
        clean_q_sample.extend(q_vals.tolist())

clean_q_sample = np.array(clean_q_sample, dtype='float32')
# Compute 5 quantile bin edges
q_bin_edges = np.quantile(clean_q_sample, np.linspace(0, 1, 6))
q_bin_edges = np.unique(q_bin_edges)  # handle duplicates
print(f"Q-quantile bin edges: {q_bin_edges}")
del clean_q_sample
gc.collect();

In [ ]:
# ---- Step 2: collect distance values per q-bin for boxplot ----
# We store sampled distance values per bin (max ~100k per bin).
MAX_PER_BIN = 100_000
n_q_bins = len(q_bin_edges) - 1
bin_samples = {i: [] for i in range(n_q_bins)}
rng = np.random.default_rng(42)

for chunk in iter_chunks(add_speed_cols=True):
    chunk = classify_usability(chunk)
    clean = chunk[chunk['usability_class'] == 'Inside Shop, Valid Speed (Usable Data)'].copy()
    if len(clean) == 0:
        continue
    # Assign q-bin index
    clean['q_bin_idx'] = np.digitize(clean['q'].values, q_bin_edges) - 1
    clean['q_bin_idx'] = clean['q_bin_idx'].clip(0, n_q_bins - 1)

    for bi in range(n_q_bins):
        if len(bin_samples[bi]) >= MAX_PER_BIN:
            continue
        d = clean.loc[clean['q_bin_idx'] == bi, 'distance'].dropna().values
        if len(d) == 0:
            continue
        take = min(len(d), MAX_PER_BIN - len(bin_samples[bi]))
        if take < len(d):
            idx = rng.choice(len(d), size=take, replace=False)
            d = d[idx]
        bin_samples[bi].extend(d.tolist())

# Create boxplot
bin_labels = [f"{q_bin_edges[i]:.0f}–{q_bin_edges[i+1]:.0f}" for i in range(n_q_bins)]
box_data = [np.array(bin_samples[i]) for i in range(n_q_bins)]

plt.figure(figsize=(10, 6))
bp = plt.boxplot(box_data, labels=bin_labels, patch_artist=True, showfliers=False)
plt.title('Fully Cleaned Data: Boxplot of Distance (Drift) by Quality (q) Bins')
plt.xlabel('q Bins')
plt.ylabel('Distance')
plt.yscale('log')
plt.show()

del bin_samples
gc.collect();

## 5. Positioning Accuracy using Stationary Devices (Paikannustarkkuus)
We identify periods when the device is stationary (e.g. speed < 5) and calculate the standard deviation of `x` and `y`.

In [ ]:
# ---- Streaming computation of stationary-point statistics ----
stat_count = 0
stat_x_sum = 0.0
stat_x_sum2 = 0.0
stat_y_sum = 0.0
stat_y_sum2 = 0.0
stat_dist_sum = 0.0
stat_dist_count = 0

# For per-q_rounded aggregation, collect sums and counts
q_round_agg = {}  # q_rounded → {n, x_sum, x_sum2, y_sum, y_sum2}

for chunk in iter_chunks(add_speed_cols=True):
    chunk = classify_usability(chunk)
    clean = chunk[chunk['usability_class'] == 'Inside Shop, Valid Speed (Usable Data)']
    stat = clean[clean['speed'] < 5.0].copy()
    if len(stat) == 0:
        continue

    stat_count += len(stat)

    xd = stat['x_diff'].dropna().values.astype('float64')
    yd = stat['y_diff'].dropna().values.astype('float64')
    stat_x_sum  += xd.sum()
    stat_x_sum2 += (xd ** 2).sum()
    stat_y_sum  += yd.sum()
    stat_y_sum2 += (yd ** 2).sum()

    dd = stat['distance'].dropna().values.astype('float64')
    stat_dist_sum  += dd.sum()
    stat_dist_count += len(dd)

    # Per q_rounded
    stat['q_rounded'] = stat['q'].round(-1)
    for qr, grp in stat.groupby('q_rounded'):
        xg = grp['x_diff'].dropna().values.astype('float64')
        yg = grp['y_diff'].dropna().values.astype('float64')
        if qr not in q_round_agg:
            q_round_agg[qr] = dict(n_x=0, x_sum=0.0, x_sum2=0.0, n_y=0, y_sum=0.0, y_sum2=0.0)
        a = q_round_agg[qr]
        a['n_x']     += len(xg)
        a['x_sum']   += xg.sum()
        a['x_sum2']  += (xg ** 2).sum()
        a['n_y']     += len(yg)
        a['y_sum']   += yg.sum()
        a['y_sum2']  += (yg ** 2).sum()

# ---- Print summary ----
print(f"Number of Stationary Points: {stat_count}")
if stat_count > 0:
    x_mean = stat_x_sum / stat_count
    x_std  = np.sqrt(stat_x_sum2 / stat_count - x_mean**2)
    y_mean = stat_y_sum / stat_count
    y_std  = np.sqrt(stat_y_sum2 / stat_count - y_mean**2)
    avg_err = stat_dist_sum / stat_dist_count
    print(f"X Standard Deviation (Accuracy): {x_std:.2f}")
    print(f"Y Standard Deviation (Accuracy): {y_std:.2f}")
    print(f"Average Euclidean Error when stationary: {avg_err:.2f}")

In [ ]:
# ---- Plot: std dev vs q ----
rows = []
for qr in sorted(q_round_agg):
    a = q_round_agg[qr]
    if a['n_x'] > 1 and a['n_y'] > 1:
        x_m = a['x_sum'] / a['n_x']
        x_s = np.sqrt(a['x_sum2'] / a['n_x'] - x_m**2)
        y_m = a['y_sum'] / a['n_y']
        y_s = np.sqrt(a['y_sum2'] / a['n_y'] - y_m**2)
        rows.append({'q_rounded': qr, 'x_std': x_s, 'y_std': y_s})

std_by_q = pd.DataFrame(rows)

plt.figure(figsize=(10, 5))
plt.plot(std_by_q['q_rounded'], std_by_q['x_std'], label='X Standard Deviation', marker='o')
plt.plot(std_by_q['q_rounded'], std_by_q['y_std'], label='Y Standard Deviation', marker='s')
plt.title('Positioning Standard Deviation vs Quality (q) [Stationary Devices]')
plt.xlabel('Quality (q)')
plt.ylabel('Standard Deviation (Distance)')
plt.legend()
plt.show()

del q_round_agg
gc.collect();
print('Done – all processing completed with chunked memory-efficient reads.')